In [1]:
import anthropic
import os
from sentence_transformers import SentenceTransformer
import lancedb

print("All imports successful")

All imports successful


## Load Embedding Model

Same `all-MiniLM-L6-v2` model used in notebook 05 to create the stored embeddings. We need it here to encode user questions at query time.

In [2]:
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded")
print(f"  Model: all-MiniLM-L6-v2")
print(f"  Embedding dimension: 384")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
  Model: all-MiniLM-L6-v2
  Embedding dimension: 384


## Load Pre-Created Embeddings from LanceDB

Connects to the LanceDB instance and opens the `entities` table populated in notebook 05.

In [3]:
db = lancedb.connect('/workspace/lancedb_data')
table = db.open_table('entities')

print(f"Connected to LanceDB")
print(f"Total entities available: {table.count_rows()}")

# Preview the data
print("\nSample data from LanceDB:")
sample = table.to_pandas().head(5)
display_columns = ['entity_id', 'name', 'type', 'data_sources', 'num_records', 'risks']
print(sample[display_columns].to_string())

Connected to LanceDB
Total entities available: 196

Sample data from LanceDB:
   entity_id                               name          type                   data_sources  num_records               risks
0     100001                    Abassin Badshah        PERSON  OPEN-OWNERSHIP,OPEN-SANCTIONS            3        corp.disqual
1     100002                        LMAR GB LTD  ORGANIZATION                 OPEN-SANCTIONS            1                    
2     100003            WANDLE HOLDINGS LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1     sanction.linked
3     100004  POLYUS GOLD INTERNATIONAL LIMITED  ORGANIZATION                 OPEN-SANCTIONS            1     sanction.linked
4     100005          Firuza Nazimovna Kerimova        PERSON                 OPEN-SANCTIONS            1  role.rca; sanction


## Set Up Anthropic Client

In [4]:
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found in environment")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic client configured with Claude Sonnet")

Anthropic client configured with Claude Sonnet


## RAG Query Function

Simple vector-only RAG pipeline: encode the question, search LanceDB for the most similar entities, assemble context from the results, and ask Claude. No graph expansion — retrieval is purely based on vector similarity.

In [5]:
def ask_simple_rag(question, top_k=10):
    """
    Simple RAG: Vector search -> Format context -> Ask LLM
    No knowledge graph — retrieval is purely vector similarity.
    """
    print(f"\nQuestion: {question}")
    print("="*70)
    
    # Step 1: Vector search
    question_embedding = embedding_model.encode(question).tolist()
    results = table.search(question_embedding).limit(top_k).to_list()
    
    print(f"Found {len(results)} relevant entities")
    
    # Step 2: Build context from search results only
    context_parts = ["ENTITIES:\n"]
    
    for info in results:
        context_parts.append(f"- {info['name']} ({info['type']})")
        context_parts.append(f"  Sources: {info['data_sources']}, Records: {info['num_records']}")
        
        if info.get('addresses'):
            context_parts.append(f"  Addresses: {info['addresses']}")
        
        if info.get('identifiers'):
            context_parts.append(f"  Identifiers: {info['identifiers']}")
        
        if info.get('risks'):
            context_parts.append(f"  Risks: {info['risks']}")
        
        context_parts.append("")
    
    context = "\n".join(context_parts)
    
    # Step 3: Ask LLM
    print("Querying LLM...")
    
    message = client.messages.create(
        model="claude-sonnet-4-5-20250929",
        max_tokens=2048,
        system="Answer questions about a corporate ownership and sanctions dataset. You only have access to individual entity records — you do not have relationship or graph data.",
        messages=[
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]
    )
    
    print("\n" + "="*70)
    print("ANSWER")
    print("="*70)
    print(message.content[0].text)
    print("="*70)

## Interactive Query Session

In [ ]:
print("Simple RAG (No Knowledge Graph) - Interactive Session")
print("="*70)
print("Ask any question about the corporate ownership and sanctions data.")
print("Retrieval is vector similarity only — no graph relationships.")
print("Type 'quit' to exit.\n")

while True:
    question = input("Your question: ").strip()
    
    if question.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    
    if not question:
        continue
    
    try:
        ask_simple_rag(question)
        print()
        
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Simple RAG (No Knowledge Graph) - Interactive Session
Ask any question about the corporate ownership and sanctions data.
Retrieval is vector similarity only — no graph relationships.
Type 'quit' to exit.



Your question:  tell me about nugget capital



Question: tell me about nugget capital
Found 10 relevant entities
Querying LLM...

ANSWER
# NUGGET CAPITAL PLC

Based on the available data, here's what we know about Nugget Capital PLC:

## Basic Information
- **Type**: Organization (Company)
- **Registration**: GB-COH: 07706746 (UK Companies House)
- **Location**: 8th Floor, 20 Farringdon Street, London, EC4A 4AB

## Data Sources
The entity appears in multiple databases:
- **OPEN-OWNERSHIP**: Corporate ownership records
- **OPEN-SANCTIONS**: Sanctions and risk databases

## Risk Profile
⚠️ **IMPORTANT**: Nugget Capital PLC has a **sanctions-related risk flag**:
- **Risk Category**: `sanction.linked`

This means the entity is linked to sanctions in some way, though the exact nature of the connection (whether directly sanctioned, owned by sanctioned parties, or associated with sanctioned entities) would require additional relationship data not available in these individual records.

## Additional Identifiers
- OPEN-SANCTIONS ID: rupep

Your question:  are nugget capital and said kerimov related?



Question: are nugget capital and said kerimov related?
Found 10 relevant entities
Querying LLM...

ANSWER
Based on the available data, **yes, there appears to be a connection between Nugget Capital PLC and Said Kerimov**.

Here's what the records show:

1. **NUGGET CAPITAL PLC** (ORGANIZATION)
   - Address: 8th Floor 20 Farringdon Street, London, EC4A 4AB
   - Risk designation: **sanction.linked** (indicating it's linked to sanctioned entities)

2. **Said Kerimov** (PERSON - from OPEN-OWNERSHIP source)
   - One of his addresses is: **8th Floor, 20 Farringdon Street, London, EC4A 4AB**
   - This is the **exact same address** as Nugget Capital PLC

The fact that Said Kerimov shares the same London address as Nugget Capital PLC strongly suggests a relationship between them, likely indicating ownership, directorship, or significant control. This is further supported by Nugget Capital's "sanction.linked" risk designation, which aligns with the Kerimov family's sanctions profile.

It's wort